In [ ]:
# =========================
# CELL 1 — IMPORTS + SAFETY
# =========================

import cv2
import numpy as np
import time

from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from IPython.display import display
import ipywidgets as widgets

from jetracer.nvidia_racecar import NvidiaRacecar

# --- INIT CAR ---
car = NvidiaRacecar()

# --- HARD SAFETY STOP ---
car.throttle = 0.0
car.steering = 0.0

print("System initialised. Car STOPPED.")

In [ ]:
# =========================
# CELL 2 — CAMERA SETUP (FIXED)
# =========================

camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True
time.sleep(2)

image_widget = widgets.Image(format='jpeg')
display(image_widget)

def update_camera():
    frame = camera.value

    if frame is None:
        print("Frame is None. Waiting...")
        time.sleep(0.5)
        frame = camera.value

    if frame is not None:
        image_widget.value = bgr8_to_jpeg(frame)

    return frame

frame = update_camera()

print("Camera ready.")
print("Frame is None:", frame is None)

if frame is not None:
    print("Shape:", frame.shape)
    print("Min:", frame.min())
    print("Max:", frame.max())
    print("Mean:", round(frame.mean(), 2))

In [ ]:
# =========================
# CELL 3 — CALIBRATED MOTION
# =========================

FORWARD_THROTTLE = -0.22
REVERSE_THROTTLE = 0.20
KICK_THROTTLE = -0.28

STEERING_OFFSET = 0.03
MAX_STEER = 0.70

def stop(t=0.15):
    car.throttle = 0.0
    car.steering = 0.0
    time.sleep(t)

def drive(throttle, steering, duration):
    car.steering = float(np.clip(steering + STEERING_OFFSET, -MAX_STEER, MAX_STEER))
    car.throttle = float(throttle)
    time.sleep(duration)
    stop()

def forward(duration=0.30, steering=0.0):
    car.steering = float(np.clip(steering + STEERING_OFFSET, -MAX_STEER, MAX_STEER))
    car.throttle = KICK_THROTTLE
    time.sleep(0.12)
    car.throttle = FORWARD_THROTTLE
    time.sleep(duration)
    stop()

def reverse(duration=0.25, steering=0.0):
    drive(REVERSE_THROTTLE, steering, duration)

def arc_left(duration=0.35, steer=0.55):
    forward(duration, steering=-steer)

def arc_right(duration=0.35, steer=0.55):
    forward(duration, steering=steer)

def reverse_left(duration=0.30, steer=0.55):
    reverse(duration, steering=-steer)

def reverse_right(duration=0.30, steer=0.55):
    reverse(duration, steering=steer)

print("Calibrated motion ready.")

In [ ]:
# =========================
# CELL 4 — WHITE MASK
# =========================

def get_white_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Strict mask
    lower1 = np.array([0, 0, 200])
    upper1 = np.array([180, 40, 255])
    mask1 = cv2.inRange(hsv, lower1, upper1)

    # Fallback mask (brighter tolerance)
    lower2 = np.array([0, 0, 170])
    upper2 = np.array([180, 70, 255])
    mask2 = cv2.inRange(hsv, lower2, upper2)

    mask = mask1

    # If too few pixels → fallback
    if np.sum(mask1 > 0) < 400:
        mask = mask2

    # Morphological cleanup
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    return mask

In [ ]:
# =========================
# CELL 5 — MASK DEBUG
# =========================

frame = update_camera()
mask = get_white_mask(frame)

white_pixels = np.sum(mask > 0)
ratio = white_pixels / (224*224)

print("White pixels:", white_pixels)
print("White ratio:", round(ratio, 4))

debug = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
image_widget.value = bgr8_to_jpeg(debug)

In [ ]:
# =========================
# CELL 6 — BAY GEOMETRY DETECTOR v2
# =========================
# Uses Hough line detection instead of only vertical contours.
# This handles slanted parking-bay side lines much better.

def detect_bay(frame):
    height, width, _ = frame.shape
    image_center = width // 2

    mask = get_white_mask(frame)
    debug = frame.copy()

    # Draw camera centre
    cv2.line(debug, (image_center, 0), (image_center, height), (0, 0, 255), 2)

    # Clean mask slightly more for line detection
    kernel = np.ones((3, 3), np.uint8)
    clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Detect line segments
    lines = cv2.HoughLinesP(
        clean,
        rho=1,
        theta=np.pi / 180,
        threshold=18,
        minLineLength=20,
        maxLineGap=12
    )

    candidates = []

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]

            dx = x2 - x1
            dy = y2 - y1
            length = np.sqrt(dx * dx + dy * dy)

            if length < 20:
                continue

            # Reject mostly horizontal top tape
            if abs(dy) < abs(dx) * 0.5:
                continue

            # Bottom point = closer entrance point
            if y1 > y2:
                bottom_x, bottom_y = x1, y1
                top_x, top_y = x2, y2
            else:
                bottom_x, bottom_y = x2, y2
                top_x, top_y = x1, y1

            mid_x = (x1 + x2) // 2
            mid_y = (y1 + y2) // 2

            candidates.append({
                "bottom_x": int(bottom_x),
                "bottom_y": int(bottom_y),
                "top_x": int(top_x),
                "top_y": int(top_y),
                "mid_x": int(mid_x),
                "mid_y": int(mid_y),
                "length": float(length),
                "dx": int(dx),
                "dy": int(dy)
            })

            cv2.line(debug, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(debug, (int(bottom_x), int(bottom_y)), 5, (0, 255, 255), -1)

    found = False
    center = None
    angle = None
    bay_width = None
    mode = "NO_BAY"

    # Choose best left and right side lines
    if len(candidates) >= 2:
        left_candidates = [c for c in candidates if c["bottom_x"] < image_center]
        right_candidates = [c for c in candidates if c["bottom_x"] >= image_center]

        if len(left_candidates) > 0 and len(right_candidates) > 0:
            left = max(left_candidates, key=lambda c: c["length"])
            right = max(right_candidates, key=lambda c: c["length"])

            bay_width = right["bottom_x"] - left["bottom_x"]

            if 35 <= bay_width <= 180:
                cx = (left["bottom_x"] + right["bottom_x"]) // 2
                cy = (left["bottom_y"] + right["bottom_y"]) // 2

                center = (cx, cy)

                # Angle proxy: if one entrance point is lower than the other, robot is angled
                angle = right["bottom_y"] - left["bottom_y"]

                found = True
                mode = "FULL_BAY"

                cv2.circle(debug, (left["bottom_x"], left["bottom_y"]), 8, (0, 255, 0), -1)
                cv2.circle(debug, (right["bottom_x"], right["bottom_y"]), 8, (0, 255, 0), -1)
                cv2.line(debug, (left["bottom_x"], left["bottom_y"]),
                         (right["bottom_x"], right["bottom_y"]), (255, 255, 0), 2)

                cv2.circle(debug, center, 8, (255, 0, 0), -1)
                cv2.line(debug, (cx, 0), (cx, height), (255, 0, 0), 2)

    # Partial fallback
    if not found and len(candidates) == 1:
        c = candidates[0]
        found = True
        mode = "PARTIAL"
        center = (c["bottom_x"], c["bottom_y"])
        angle = None
        bay_width = None

        cv2.circle(debug, center, 8, (0, 165, 255), -1)

    cv2.putText(debug, mode, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                (255, 0, 0) if mode == "FULL_BAY" else (0, 165, 255),
                2)

    cv2.putText(debug, f"lines:{len(candidates)} width:{bay_width} angle:{angle}",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                (0, 255, 255), 1)

    return found, center, angle, bay_width, debug

In [ ]:
# =========================
# CELL 7 — TEST BAY DETECTOR
# =========================
# PURPOSE:
# This cell checks whether the bay detector from Cell 6 is still working.
# It should detect:
# - found: whether the parking bay is visible
# - center: estimated entrance centre of the bay
# - angle: rough entrance tilt
# - width: distance between the left and right bay entrance points
#
# This cell DOES NOT move the car.

frame = update_camera()

if frame is None:
    print("No frame detected.")
else:
    found, center, angle, width, debug = detect_bay(frame)

    print("Found:", found)
    print("Center:", center)
    print("Angle:", angle)
    print("Width:", width)

    image_widget.value = bgr8_to_jpeg(debug)

In [ ]:
# =========================
# CELL 8 — STABILISED BAY SCAN
# =========================

def scan_bay(num_frames=6, delay=0.06):
    centers = []
    angles = []
    widths = []
    debug_last = None
    full_count = 0
    partial_count = 0

    for _ in range(num_frames):
        frame = camera.value
        if frame is None:
            time.sleep(delay)
            continue

        found, center, angle, width, debug = detect_bay(frame)
        debug_last = debug

        if found and center is not None and width is not None:
            full_count += 1
            centers.append(center)
            angles.append(angle if angle is not None else 0)
            widths.append(width)

        elif found:
            partial_count += 1

        time.sleep(delay)

    if full_count > 0:
        avg_x = int(np.mean([c[0] for c in centers]))
        avg_y = int(np.mean([c[1] for c in centers]))
        avg_angle = int(np.mean(angles))
        avg_width = int(np.mean(widths))
        confidence = full_count / num_frames

        return {
            "mode": "FULL",
            "found": True,
            "center": (avg_x, avg_y),
            "angle": avg_angle,
            "width": avg_width,
            "confidence": confidence,
            "debug": debug_last
        }

    if partial_count > 0:
        return {
            "mode": "PARTIAL",
            "found": True,
            "center": None,
            "angle": None,
            "width": None,
            "confidence": partial_count / num_frames,
            "debug": debug_last
        }

    return {
        "mode": "NONE",
        "found": False,
        "center": None,
        "angle": None,
        "width": None,
        "confidence": 0.0,
        "debug": debug_last
    }

print("Stabilised scan ready.")

In [ ]:
# =========================
# CELL 9 — STATIC POSE TEST
# =========================

result = scan_bay(num_frames=8)

if result["debug"] is not None:
    image_widget.value = bgr8_to_jpeg(result["debug"])

print("Mode:", result["mode"])
print("Found:", result["found"])
print("Center:", result["center"])
print("Angle:", result["angle"])
print("Width:", result["width"])
print("Confidence:", round(result["confidence"], 2))

if result["center"] is not None:
    print("X error:", result["center"][0] - (camera.width // 2))
    print("Y position:", result["center"][1])

In [ ]:
# =========================
# CELL 10 — PARKING ENTRANCE CONTROLLER
# =========================

IMAGE_CENTER = camera.width // 2

X_TOL = 6
ANGLE_TOL = 10
TARGET_Y = 150
MIN_FULL_CONF = 0.45

MAX_STEPS = 45

def correct_partial():
    print("PARTIAL → reverse + sweep to recover full geometry")
    reverse(duration=0.30, steering=0.0)
    arc_right(duration=0.35, steer=0.60)

def correct_angle(angle):
    print("ANGLE correction:", angle)
    reverse(duration=0.22, steering=0.0)

    if angle < 0:
        arc_right(duration=0.35, steer=0.62)
    else:
        arc_left(duration=0.35, steer=0.62)

def correct_x(error):
    print("X correction:", error)

    if abs(error) > 25:
        duration = 0.42
        steer = 0.65
    elif abs(error) > 12:
        duration = 0.32
        steer = 0.58
    else:
        duration = 0.22
        steer = 0.45

    if error < 0:
        arc_left(duration=duration, steer=steer)
    else:
        arc_right(duration=duration, steer=steer)

def run_parking_alignment():
    print("Parking alignment controller started")

    for step in range(1, MAX_STEPS + 1):
        print("\n--- STEP", step, "---")
        stop()

        result = scan_bay(num_frames=6)

        if result["debug"] is not None:
            image_widget.value = bgr8_to_jpeg(result["debug"])

        print("Mode:", result["mode"],
              "| Center:", result["center"],
              "| Angle:", result["angle"],
              "| Width:", result["width"],
              "| Conf:", round(result["confidence"], 2))

        if result["mode"] == "NONE":
            print("SEARCH → slow scan right")
            arc_right(duration=0.25, steer=0.40)
            continue

        if result["mode"] == "PARTIAL":
            correct_partial()
            continue

        if result["confidence"] < MIN_FULL_CONF:
            print("Weak FULL confidence → recover")
            correct_partial()
            continue

        cx, cy = result["center"]
        x_error = cx - IMAGE_CENTER
        angle = result["angle"] if result["angle"] is not None else 0

        print("x_error:", x_error, "| angle:", angle, "| y:", cy)

        if abs(angle) > ANGLE_TOL:
            correct_angle(angle)
            continue

        if abs(x_error) > X_TOL:
            correct_x(x_error)
            continue

        if cy < TARGET_Y:
            print("Aligned but far → controlled approach")
            steering = np.clip(0.006 * x_error + 0.004 * angle, -0.35, 0.35)
            forward(duration=0.28, steering=steering)
            continue

        print("SUCCESS: aligned at parking bay entrance")
        stop()
        return True

    print("STOPPED: max steps reached")
    stop()
    return False

In [ ]:
# =========================
# CELL 11 — RUN CONTROLLER
# =========================

print("Starting parking alignment...\n")

success = run_parking_alignment()

if success:
    print("\n=== FINAL RESULT: SUCCESS ===")
else:
    print("\n=== FINAL RESULT: FAILED / STOPPED ===")

In [ ]:
# =========================
# CELL 12 — HARD STOP / RESET
# =========================

def emergency_stop():
    car.throttle = 0.0
    car.steering = 0.0
    time.sleep(0.2)

# Execute immediately
emergency_stop()

print("Emergency stop executed.")